# 05 — Évaluer des probabilités sans fuite temporelle

Les labels viennent uniquement de ``market_outcomes``. Les plis sont croissants : un résultat résolu après la fin d'entraînement ne peut ni choisir les joueurs ni entraîner le modèle.

In [ ]:
import numpy as np
import polars as pl
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from poly_data.analysis.ml_evaluation import expanding_folds, evaluate_edge, probability_metrics, select_fold_players
from poly_data.io.parquet_store import ParquetStore
from poly_data.notebooks import resolve_notebook_context, source_inventory

ctx = resolve_notebook_context()
store = ParquetStore(ctx.root)
print({'root': str(ctx.root), 'mode': ctx.mode, 'revision': ctx.revision})
source_inventory(store, ['trades', 'market_outcomes', 'markets_current'])

## Panel de décision et labels officiels

Pour rester strict, chaque observation est prise avant ``resolved_at``. Les marchés encore ouverts ne créent pas de ligne supervisée.

In [ ]:
outcomes = store.scan('market_outcomes').select(['market_id', 'winner_token', 'resolved_at']).collect()
trades = store.scan('trades').select(['market_id', 'timestamp', 'nonusdc_side', 'price']).collect()
snapshots = (
    trades.join(outcomes, on='market_id', how='inner')
    .filter(pl.col('timestamp') < pl.col('resolved_at') - 86_400)
    .filter(pl.col('nonusdc_side') == 'token1')
    .sort('timestamp').group_by('market_id').last()
    .select(['market_id', pl.col('timestamp').alias('decision_ts'), pl.col('price').alias('market_price'), 'winner_token', 'resolved_at'])
    .with_columns((pl.col('winner_token') == 'token1').cast(pl.Int64).alias('target'))
    .sort('decision_ts')
)
snapshots

In [ ]:
folds = expanding_folds(snapshots['decision_ts'].to_list(), n_folds=3)
print(folds)
if folds:
    selected = select_fold_players(store.scan('trades'), outcomes, folds[0].train_end_ts, n=20)
    print('players selected from known outcomes only:', selected.height)

## Baselines, régression logistique et XGBoost

Les scores sont des règles propres : log-loss et Brier. La ligne de marché est une baseline probabiliste ; la classe majoritaire est une seconde baseline.

In [ ]:
reports = []
for fold in folds:
    train = snapshots.filter(pl.col('resolved_at') < fold.train_end_ts)
    test = snapshots.filter((pl.col('decision_ts') >= fold.test_start_ts) & (pl.col('decision_ts') <= fold.test_end_ts))
    if train.height < 8 or test.height == 0 or train['target'].n_unique() < 2:
        continue
    X_train, y_train = train.select(['market_price']).to_numpy(), train['target'].to_numpy()
    X_test, y_test = test.select(['market_price']).to_numpy(), test['target'].to_numpy()
    models = {
        'market_price': np.clip(test['market_price'].to_numpy(), 1e-6, 1 - 1e-6),
        'majority': np.full(test.height, y_train.mean()),
        'logistic': LogisticRegression(random_state=7).fit(X_train, y_train).predict_proba(X_test)[:, 1],
        'xgboost': XGBClassifier(n_estimators=40, max_depth=2, learning_rate=0.1, random_state=7, eval_metric='logloss').fit(X_train, y_train).predict_proba(X_test)[:, 1],
    }
    for name, probabilities in models.items():
        reports.append({'model': name, 'fold_end': fold.test_end_ts, 'support': test.height, **probability_metrics(y_test, probabilities)})
pl.DataFrame(reports).sort(['model', 'fold_end']) if reports else pl.DataFrame()

In [ ]:
if reports:
    last_fold = folds[-1]
    train = snapshots.filter(pl.col('resolved_at') < last_fold.train_end_ts)
    test = snapshots.filter((pl.col('decision_ts') >= last_fold.test_start_ts) & (pl.col('decision_ts') <= last_fold.test_end_ts))
    model = LogisticRegression(random_state=7).fit(train.select(['market_price']).to_numpy(), train['target'].to_numpy())
    held_out = evaluate_edge(model.predict_proba(test.select(['market_price']).to_numpy())[:, 1], test['market_price'].to_numpy(), test['target'].to_numpy(), threshold=0.05)
    held_out
else:
    print('fixture too small for a fully labelled held-out fold')

Un edge positif est une comparaison de probabilités tenue à l'écart, non une preuve de rentabilité. Toute simulation de PnL doit ensuite modéliser exécution et règlement.